In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="6"

In [ ]:
import torch
import numpy as np

In [ ]:
from aidan_lib.models.sam3_base import SAM3Harness

In [ ]:
from pathlib import Path
import cv2
import imageio
from PIL import Image

In [ ]:
from aidan_lib.definitions import DATA_DIR
test_vid_path = DATA_DIR / "tip_to_tip_short.mp4"
assert test_vid_path.exists(), f"Test video does not exist {test_vid_path.absolute().as_posix()}"

In [ ]:
from aidan_lib.video_utils.load_batched_frames import load_batched_frames, load_constrained_batched_frames
from aidan_lib.video_utils.scene_split import get_constrained_scenes, get_transnet_model

In [ ]:
transnet = get_transnet_model("cuda")
constrained_scenes = get_constrained_scenes(test_vid_path, transnet, threshold=0.75)

In [ ]:
harness = SAM3Harness(compile=False, warm_up=False)

In [ ]:
batch_frame_loader = load_constrained_batched_frames(test_vid_path, constrained_scenes, batch_size=120, skip_frames=None, convert_pil=True, overlap=1)

In [ ]:
frame_batch, frame_numbers, done = next(batch_frame_loader)
print(len(frame_batch))
print(f"{frame_numbers[0]} to {frame_numbers[-1]}")
print(done)

In [ ]:
harness.start_session(video=frame_batch, frame_numbers=frame_numbers)

In [ ]:
harness.reset_session()

In [ ]:
harness.add_prompt("Person", frame_index=0)

In [ ]:
out = harness.propagate_session()

In [ ]:
harness.close_session()

In [ ]:
out.segmentation.max()

In [ ]:
from IPython.display import Image as IPyImage
from IPython.display import display

In [ ]:
from aidan_lib.visualization.segmentations import visualize_segmentations, int_mask_to_binary_masks

In [ ]:
test_frame_idx = -1
test_global_frame_index = out.video_frame_indices[test_frame_idx]

last_frame_img = frame_batch[test_frame_idx]
sam_seg = out.segmentation[test_frame_idx]

masks, obj_ids = int_mask_to_binary_masks(sam_seg, background_index=out.background_index)

visualize_segmentations(last_frame_img, masks, labels=[f"Person {obj_ids[0]}"])